# Community Detection for Fraud Ring Analysis

This notebook demonstrates graph analytics for detecting fraud communities, collusion networks, and organized financial crime patterns.

## Key Features
- **Connected Components**: Identify clusters of related accounts
- **PageRank**: Find central orchestrators in fraud networks
- **Triangle Counting**: Detect collusion patterns
- **Risk Scoring**: Community-level risk assessment

## Regulatory Context
- FinCEN SAR Reporting requirements
- FATF Recommendation 20
- EU AMLD6 criminal liability provisions

**Author**: David LECONTE - IBM Worldwide | Data & AI | Tiger Team | Data Watsonx.Data Global Product Specialist (GPS)  
**Date**: 2026-03-23

In [ ]:
# Import required libraries
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

from src.python.client.janusgraph_client import JanusGraphClient
from banking.analytics.community_detection import (
    CommunityDetector,
    CommunityRiskLevel,
    create_community_detection_report
)

print("✅ Libraries imported successfully")

## 1. Connect to JanusGraph

In [ ]:
# Connect to JanusGraph
client = JanusGraphClient(
    host="localhost",
    port=18182,
    use_ssl=False,
    verify_certs=False
)

# Verify connection
vertex_count = client.execute("g.V().count()")[0]
edge_count = client.execute("g.E().count()")[0]

print(f"Connected to JanusGraph")
print(f"Total vertices: {vertex_count}")
print(f"Total edges: {edge_count}")

## 2. Load Fraud Community Data

Load pre-generated fraud community data if not already in the graph.

In [ ]:
# Check if fraud community data exists
fraud_vertices = client.execute("g.V().has('role', 'orchestrator').count()")[0]

if fraud_vertices == 0:
    print("Loading fraud community data...")
    from banking.data_generators.scripts.generate_fraud_communities import (
        generate_fraud_communities_deterministic,
        load_into_janusgraph
    )
    
    data = generate_fraud_communities_deterministic(seed=42)
    counts = load_into_janusgraph(data)
    print(f"Loaded {counts['vertices']} vertices and {counts['edges']} edges")
else:
    print(f"Fraud community data already exists: {fraud_vertices} orchestrators found")

## 3. Explore Fraud Network Structure

In [ ]:
# Get overview of fraud network
orchestrators = client.execute("""
    g.V().has('role', 'orchestrator').
        project('id', 'name', 'riskScore').
        by('personId').by('name').by('riskScore')
""")

print(f"Found {len(orchestrators)} orchestrators:\n")
for orch in orchestrators:
    print(f"  • {orch['name']} - Risk: {orch['riskScore']:.2f}")

In [ ]:
# Get network statistics by role
role_stats = client.execute("""
    g.V().hasLabel('Person').
        group().
        by('role').
        by(project('count', 'avgRisk').by(count()).by(values('riskScore').mean()))
""")

print("Network Role Distribution:\n")
print(f"{'Role':<15} {'Count':>10} {'Avg Risk':>12}")
print("-" * 40)
for role, stats in role_stats[0].items() if role_stats else []:
    print(f"{role:<15} {stats['count']:>10} {stats['avgRisk']:>12.2f}")

## 4. Run Community Detection

In [ ]:
# Initialize community detector
detector = CommunityDetector(client, min_community_size=3)

# Run detection
print("Running community detection...\n")
results = detector.detect_fraud_communities(
    include_transactions=True,
    min_risk_score=0.3
)

print(f"Detection complete!")
print(f"Total communities: {results.total_communities}")
print(f"High-risk communities: {results.high_risk_communities}")
print(f"Total fraud actors: {results.total_fraud_actors}")
print(f"Orchestrators identified: {results.orchestrators_identified}")

## 5. Analyze Detected Communities

In [ ]:
# Display top communities by risk
print("Top Communities by Risk Score:")
print("=" * 80)

for i, community in enumerate(results.communities[:5], 1):
    print(f"\n{i}. Community {community.community_id}")
    print(f"   Members: {community.member_count}")
    print(f"   Risk Level: {community.risk_level.value.upper()}")
    print(f"   Risk Score: {community.risk_score:.2f}")
    print(f"   Total Value: ${community.total_value:,.2f}")
    print(f"   Orchestrators: {len(community.orchestrators)}")
    print(f"   Indicators: {', '.join(community.indicators[:4])}")

In [ ]:
# Detailed member analysis for top community
top_community = results.communities[0] if results.communities else None

if top_community:
    print(f"\nMember Analysis for {top_community.community_id}:")
    print("=" * 80)
    print(f"{'Name':<30} {'Role':<12} {'Degree':>8} {'Triangles':>10} {'PageRank':>10}")
    print("-" * 80)
    
    for member in sorted(top_community.members, key=lambda m: m.pagerank_score, reverse=True):
        print(f"{member.name[:28]:<30} {member.role:<12} {member.degree:>8} {member.triangle_count:>10} {member.pagerank_score:>10.4f}")

## 6. Visualize Fraud Communities

In [ ]:
# Create network visualization
def visualize_community(community, ax=None):
    """Visualize a fraud community using NetworkX."""
    # Build NetworkX graph
    G = nx.Graph()
    
    # Color map for roles
    role_colors = {
        'orchestrator': '#FF4444',
        'hub': '#FF8800',
        'mule': '#FFAA00',
        'participant': '#44AA44',
        'peripheral': '#888888'
    }
    
    # Add nodes
    for member in community.members:
        G.add_node(
            member.vertex_id,
            name=member.name,
            role=member.role,
            degree=member.degree,
            color=role_colors.get(member.role, '#888888')
        )
    
    # Add edges
    for member in community.members:
        neighbors = client.execute(f"""g.V().has('personId', '{member.vertex_id}').both('knows').id()""")
        for neighbor_id in neighbors:
            neighbor_str = str(neighbor_id)
            if neighbor_str in [m.vertex_id for m in community.members]:
                G.add_edge(member.vertex_id, neighbor_str)
    
    # Create visualization
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))
    
    # Layout
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Draw nodes
    node_colors = [G.nodes[n]['color'] for n in G.nodes()]
    node_sizes = [max(100, G.nodes[n]['degree'] * 50) for n in G.nodes()]
    
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes, alpha=0.8)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, width=1)
    
    # Labels for orchestrators only
    labels = {
        n: G.nodes[n]['name'].split()[0] 
        for n in G.nodes() 
        if G.nodes[n]['role'] == 'orchestrator'
    }
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=8)
    
    ax.set_title(f"{community.community_id}\nRisk: {community.risk_level.value.upper()} ({community.risk_score:.2f})")
    ax.axis('off')
    
    return G

# Visualize top 3 communities
if results.communities:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, community in enumerate(results.communities[:3]):
        visualize_community(community, ax=axes[i])
    
    # Legend
    legend_patches = [
        mpatches.Patch(color='#FF4444', label='Orchestrator'),
        mpatches.Patch(color='#FF8800', label='Hub'),
        mpatches.Patch(color='#FFAA00', label='Mule'),
        mpatches.Patch(color='#44AA44', label='Participant'),
        mpatches.Patch(color='#888888', label='Peripheral')
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02))
    
    plt.tight_layout()
    plt.savefig('exports/community_detection_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Visualization saved to exports/community_detection_visualization.png")

## 7. PageRank Analysis - Find Key Orchestrators

In [ ]:
# Find top orchestrators by PageRank (degree-based approximation)
top_actors = client.execute("""
    g.V().hasLabel('Person').
        order().by(bothE().count(), desc).
        limit(10).
        project('id', 'name', 'role', 'degree', 'riskScore').
        by('personId').
        by('name').
        by('role').
        by(bothE().count()).
        by('riskScore')
""")

print("Top 10 Key Actors by Network Centrality:")
print("=" * 90)
print(f"{'Name':<30} {'Role':<12} {'Degree':>8} {'Risk Score':>12} {'Centrality':>12}")
print("-" * 90)

total_degree = sum(a['degree'] for a in top_actors) if top_actors else 1
for actor in top_actors:
    centrality = actor['degree'] / total_degree
    print(f"{actor['name'][:28]:<30} {actor['role']:<12} {actor['degree']:>8} {actor['riskScore']:>12.2f} {centrality:>12.4f}")

## 8. Triangle Detection - Collusion Patterns

In [ ]:
# Find triangles (potential collusion groups)
print("Detecting collusion triangles...\n")

# Simplified triangle detection using knows relationships
persons = client.execute("g.V().hasLabel('Person').has('role').id()")

triangles_found = []
for person_id in persons[:20]:  # Limit for demo
    neighbors = client.execute(f"""g.V('{person_id}').both('knows').id()""")
    for i, n1 in enumerate(neighbors):
        for n2 in neighbors[i+1:]:
            # Check if n1 and n2 are connected
            connected = client.execute(f"""g.V('{n1}').both('knows').where(eq('{n2}')).limit(1)""")
            if connected:
                triangle = tuple(sorted([str(person_id), str(n1), str(n2)]))
                if triangle not in triangles_found:
                    triangles_found.append(triangle)

print(f"Found {len(triangles_found)} triangle patterns (collusion indicators)")

if triangles_found:
    print("\nSample triangles:")
    for i, tri in enumerate(triangles_found[:5], 1):
        print(f"  {i}. {tri[0][:12]} - {tri[1][:12]} - {tri[2][:12]}")

## 9. Generate Compliance Report

In [ ]:
# Generate detailed compliance report
report = create_community_detection_report(results, output_format="detailed")
print(report)

In [ ]:
# Generate SAR-ready report
sar_report = create_community_detection_report(results, output_format="sar_ready")
print(sar_report)

## 10. Export for Visualization Tools

In [ ]:
# Export top community for D3.js / Gephi visualization
if results.communities:
    top_community = results.communities[0]
    viz_data = detector.export_for_visualization(top_community, format="json")
    
    # Save to file
    import json
    from pathlib import Path
    
    export_dir = Path("exports")
    export_dir.mkdir(exist_ok=True)
    
    with open(export_dir / f"community_{top_community.community_id}.json", "w") as f:
        json.dump(viz_data, f, indent=2)
    
    print(f"✅ Exported visualization data for {top_community.community_id}")
    print(f"   Nodes: {len(viz_data['nodes'])}")
    print(f"   Edges: {len(viz_data['edges'])}")
    print(f"   File: exports/community_{top_community.community_id}.json")

## 11. Summary Statistics

In [ ]:
# Create summary DataFrame
summary_data = {
    'Metric': [
        'Total Communities',
        'High-Risk Communities',
        'Total Fraud Actors',
        'Orchestrators Identified',
        'Mule Accounts Identified',
        'Triangle Patterns Found',
        'Average Community Size',
        'Average Risk Score'
    ],
    'Value': [
        results.total_communities,
        results.high_risk_communities,
        results.total_fraud_actors,
        results.orchestrators_identified,
        results.mule_accounts_identified,
        len(triangles_found),
        f"{results.total_fraud_actors / max(results.total_communities, 1):.1f}",
        f"{sum(c.risk_score for c in results.communities) / max(len(results.communities), 1):.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 50)
print("COMMUNITY DETECTION SUMMARY")
print("=" * 50)
print(summary_df.to_string(index=False))

## Key Takeaways

1. **Connected Components** identify clusters of related accounts that may represent fraud rings
2. **PageRank/centrality** measures help identify key orchestrators in fraud networks
3. **Triangle counting** reveals collusion patterns where three actors mutually transact
4. **Risk scoring** prioritizes communities for investigation
5. **Visualization exports** support compliance reporting and stakeholder presentations

## Next Steps
- Integrate with real-time Pulsar streaming for live detection
- Add temporal analysis for evolving fraud patterns
- Connect to OpenSearch for entity resolution across data sources

In [ ]:
# Close connection
client.close()
print("\n✅ Session complete. Connection closed.")